#### Bipedal Walker

Goal & rewards: move right across rough terrain; forward progress gives reward, energy use is slightly penalized, falling is −100. “Solved” is ≈ 300 return (normal) within the time limit. 
Gymnasium
+1

Variants: BipedalWalker-v3 (normal) and BipedalWalkerHardcore-v3 (obstacles). Try the normal one first; Hardcore is much tougher. 
Gymnasium

Gotchas: if Box2D import fails, re-check swig + gymnasium[box2d] install (Windows may need build tools). 
Gymnasium
+1

In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

In [2]:


def make_env():
    return gym.make("CarRacing-v3", continuous=True)

if __name__ == "__main__":
    num_envs = 4
    vec_env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)
    eval_env = make_vec_env(make_env, n_envs=1)

    # verbose=0 silences evaluation console output
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path="./logs/best_model",
        log_path="./logs/results",
        eval_freq=max(10_000 // num_envs, 1),
        n_eval_episodes=5,
        deterministic=True,
        render=False,
        verbose=0,
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=max(50_000 // num_envs, 1),
        save_path="./logs/checkpoints",
        name_prefix="carracing_ppo",
        verbose=0,
    )

    # verbose=0 silences the standard rollout/train summary tables
    model = PPO(
        policy="CnnPolicy",
        env=vec_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        verbose=0,
        tensorboard_log="./logs/tb/",
    )

    print("Training in progress...")
    # progress_bar=True renders a real-time progress bar across total_timesteps
    model.learn(
        total_timesteps=2_000_000,
        callback=[eval_callback, checkpoint_callback],
        progress_bar=True,
    )

    model.save("carracing_ppo_final")
    print("Training finished and model saved as carracing_ppo_final.zip")

    vec_env.close()
    eval_env.close()

Output()

Training in progress...


/Users/jagannathrao/Library/CloudStorage/OneDrive-UniversityofGeorgia/UGA_Courses/INFO_NEW_COURSES_FROM_FALL_2022/INFO_4000/INFO_4000_Fall_2026/Weekly_Content/.venv312/lib/python3.12/site-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x10eaff140> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x10f4f3830>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Training finished and model saved as carracing_ppo_final.zip


In [3]:
import gymnasium as gym
from stable_baselines3 import PPO

# 1. Initialize environment with human rendering
env = gym.make("CarRacing-v3", continuous=True, render_mode="human")

# 2. Load model weights (can also load './logs/best_model/best_model.zip')
model = PPO.load("carracing_ppo_final", env=env)

# 3. Run evaluation episodes
episodes = 5
for ep in range(episodes):
    obs, info = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        # deterministic=True ensures the policy picks the mean action without random noise
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        done = terminated or truncated

    print(f"Episode {ep + 1}: Total Reward = {total_reward:.2f}")

env.close()

Episode 1: Total Reward = 235.89
Episode 2: Total Reward = 344.44
Episode 3: Total Reward = 187.13
Episode 4: Total Reward = 368.65
Episode 5: Total Reward = 44.02
